# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de telemetría IoT

Este notebook trabaja con el nuevo Bronze oficial, guardado como `andinalog_iot_telemetry.csv` en `datasets/AndinaLog_03B_Bronce/`. Conserva los diez campos originales y registra problemas sin corregirlos. Las conversiones usadas para validar son temporales. El tratamiento corresponde al notebook 2.

Señala identificadores mal formados, lecturas en °F pendientes de estandarización, unidades no reconocidas, el centinela `-999`, y separa duplicados idénticos de lecturas con la misma clave pero contenido diferente. En los conflictos de clave marca todas las filas involucradas; no elige una lectura automáticamente.

Cada ejecución reemplaza cuatro archivos en `S4/andinalog_iot_telemetry/notebook1/salidas/`: el diagnosticado, el detalle de problemas, la cuarentena y el reporte de calidad con SHA-256 del Bronze.


## 1 · Configuración y origen

En local, ejecuta el notebook desde cualquier carpeta dentro del proyecto. En Colab, monta Drive, selecciona `ENTORNO = "drive"` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`. Las salidas van a `S4/andinalog_iot_telemetry/notebook1/salidas/` en ese mismo entorno.


In [7]:
from pathlib import Path
import hashlib
import os
import re
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"  # ajustar si la carpeta real es otra
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_iot_telemetry.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-IOT-diagnostico-v4"

COLUMNAS_ORIGINALES = [
    "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "proyecto-integrador").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "proyecto-integrador" / "andinalog_iot_telemetry"/ "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_iot_telemetry.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_iot_telemetry\notebook1\salidas


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales: no se exportan como valores transformados.


In [8]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


Bronze: 28,920 filas × 10 columnas
SHA-256: edb7afe2f7fb836e59fe605d30c88b3b5b13a6d8ab2ec0b37f206a14e58de6bf


,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,temp_unit,humedad_cabina_pct,desviacion_termica_flag,desviacion_proximos_60min_flag
0,2026-08-11 14:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.19,C,72.6,0,0
1,2026-08-11 14:41:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.91,C,75.0,0,0
2,2026-08-11 15:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.76,C,94.7,0,0
3,2026-08-11 15:41:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,3.96,C,74.0,0,0
4,2026-08-11 16:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.49,C,70.3,0,0


## 3 · Catálogo y reglas de diagnóstico

Los códigos identifican el problema exacto para el notebook 2. `DUPLICADO_IDENTICO` marca las copias posteriores de una fila exactamente repetida; `CLAVE_EN_CONFLICTO` marca todas las lecturas de una misma clave `viaje_id+timestamp` cuando sus contenidos difieren. Una lectura en `F` es válida como unidad, pero queda señalada como pendiente de estandarización. `K` permanece como unidad no reconocida hasta definir una regla. `-999` es un centinela inválido, no una temperatura utilizable.

El diagnóstico no confirma la coherencia clínica u operativa de las banderas térmicas ni interpreta la ventana futura de 60 minutos: esas reglas necesitan los umbrales por producto y una definición temporal aprobada.


In [9]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("timestamp", "FECHA_INVALIDA", "Formato incorrecto o fecha imposible"),
    *[(c, "FALTANTE", "Identificador vacío") for c in ["viaje_id", "order_id", "camion_id", "producto_id"]],
    *[(c, "FORMATO_INVALIDO", "Identificador con formato inesperado") for c in ["viaje_id", "order_id", "camion_id", "producto_id"]],
    ("viaje_id+timestamp", "DUPLICADO_IDENTICO", "Copia posterior de una fila idéntica"),
    ("viaje_id+timestamp", "CLAVE_EN_CONFLICTO", "Lecturas diferentes con la misma clave; se marcan todas"),
    ("temp_unit", "UNIDAD_NO_RECONOCIDA", "Unidad distinta de C o F"),
    ("temp_unit", "PENDIENTE_CONVERSION_F", "Lectura válida en F; pendiente de estandarizar a C"),
    ("temperatura_cabina_c", "FALTANTE", "Campo vacío"),
    ("temperatura_cabina_c", "NO_NUMERICA", "Valor no convertible a número"),
    ("temperatura_cabina_c", "VALOR_CENTINELA", "-999 indica ausencia o error de lectura"),
    ("humedad_cabina_pct", "FALTANTE", "Campo vacío"),
    ("humedad_cabina_pct", "NO_NUMERICA", "Valor no convertible a número"),
    ("humedad_cabina_pct", "FUERA_RANGO", "Humedad relativa fuera de 0 a 100%"),
    ("desviacion_termica_flag", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
    ("desviacion_proximos_60min_flag", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def detectar_identificadores(df):
    patrones = {
        "viaje_id": r"VIA-\d{5}",
        "order_id": r"ORD-\d{4}-\d{5}",
        "camion_id": r"CAM-\d{2}",
        "producto_id": r"PROD-\d{3}",
    }
    hallazgos = []
    for col, patron in patrones.items():
        valor = texto(df, col)
        hallazgos.append(registrar_problema(df, valor.eq(""), col, "FALTANTE"))
        hallazgos.append(registrar_problema(df, valor.ne("") & ~df[col].str.fullmatch(patron).fillna(False), col, "FORMATO_INVALIDO"))
    return hallazgos

def detectar_timestamps_y_duplicados(df):
    ts = texto(df, "timestamp")
    formato = ts.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").fillna(False)
    fecha = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    invalida = ~formato | fecha.isna()
    clave = pd.DataFrame({"viaje_id": texto(df, "viaje_id"), "timestamp": ts})
    # Una firma de todos los campos Bronze distingue copias de contenidos distintos.
    firma = df[COLUMNAS_ORIGINALES].astype("string").agg("\x1f".join, axis=1)
    variantes = firma.groupby([clave["viaje_id"], clave["timestamp"]], dropna=False).transform("nunique")
    conflicto = clave.duplicated(keep=False) & variantes.gt(1)
    identico = df[COLUMNAS_ORIGINALES].duplicated(keep="first") & ~conflicto
    evidencia_clave = clave["viaje_id"] + " + " + clave["timestamp"]
    return [
        registrar_problema(df, invalida, "timestamp", "FECHA_INVALIDA"),
        registrar_problema(df, identico, "viaje_id+timestamp", "DUPLICADO_IDENTICO", evidencia_clave),
        registrar_problema(df, conflicto, "viaje_id+timestamp", "CLAVE_EN_CONFLICTO", evidencia_clave),
    ]

def detectar_medicion(df, columna, rango=None, centinelas=()):
    valor = texto(df, columna)
    numero = pd.to_numeric(valor, errors="coerce")
    hallazgos = [
        registrar_problema(df, valor.eq(""), columna, "FALTANTE"),
        registrar_problema(df, valor.ne("") & numero.isna(), columna, "NO_NUMERICA"),
    ]
    if centinelas:
        hallazgos.append(registrar_problema(df, numero.isin(centinelas), columna, "VALOR_CENTINELA"))
    if rango is not None:
        minimo, maximo = rango
        hallazgos.append(registrar_problema(df, numero.notna() & ~numero.between(minimo, maximo), columna, "FUERA_RANGO"))
    return hallazgos

def detectar_unidad_y_flags(df):
    unidad = texto(df, "temp_unit").str.upper()
    hallazgos = [
        registrar_problema(df, ~unidad.isin(["C", "F"]), "temp_unit", "UNIDAD_NO_RECONOCIDA"),
        registrar_problema(df, unidad.eq("F"), "temp_unit", "PENDIENTE_CONVERSION_F"),
    ]
    for col in ["desviacion_termica_flag", "desviacion_proximos_60min_flag"]:
        valor = pd.to_numeric(texto(df, col), errors="coerce")
        hallazgos.append(registrar_problema(df, ~valor.isin([0, 1]), col, "FLAG_INVALIDA"))
    return hallazgos

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = (
        detectar_identificadores(principal)
        + detectar_timestamps_y_duplicados(principal)
        + detectar_medicion(principal, "temperatura_cabina_c", centinelas=(-999,))
        + detectar_medicion(principal, "humedad_cabina_pct", rango=(0, 100))
        + detectar_unidad_y_flags(principal)
    )
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(
        lambda valores: "|".join(dict.fromkeys(valores))
    )
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())


,columna_afectada,codigo_error,criterio
0,timestamp,FECHA_INVALIDA,Formato incorrecto o fecha imposible
1,viaje_id,FALTANTE,Identificador vacío
2,order_id,FALTANTE,Identificador vacío
3,camion_id,FALTANTE,Identificador vacío
4,producto_id,FALTANTE,Identificador vacío
5,viaje_id,FORMATO_INVALIDO,Identificador con formato inesperado
6,order_id,FORMATO_INVALIDO,Identificador con formato inesperado
7,camion_id,FORMATO_INVALIDO,Identificador con formato inesperado
8,producto_id,FORMATO_INVALIDO,Identificador con formato inesperado
9,viaje_id+timestamp,DUPLICADO_IDENTICO,Copia posterior de una fila idéntica


Principal: 28,920; problemas: 560; filas en cuarentena: 554


,columna_afectada,codigo_error,filas
0,camion_id,FORMATO_INVALIDO,50
1,humedad_cabina_pct,FALTANTE,100
2,humedad_cabina_pct,FUERA_RANGO,15
3,temp_unit,PENDIENTE_CONVERSION_F,50
4,temp_unit,UNIDAD_NO_RECONOCIDA,5
5,temperatura_cabina_c,FALTANTE,80
6,temperatura_cabina_c,VALOR_CENTINELA,120
7,timestamp,FECHA_INVALIDA,15
8,viaje_id+timestamp,CLAVE_EN_CONFLICTO,10
9,viaje_id+timestamp,DUPLICADO_IDENTICO,115


## 4 · Reporte y comprobaciones antes de exportar

El reporte registra la huella SHA-256 para reconocer la versión exacta del CSV de origen. Los conteos de problemas pueden superar el número de filas en cuarentena porque una fila puede tener varios hallazgos.


In [10]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name),
        ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(
        CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)
    ).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_iot_telemetry.csv
1,sha256_bronze,edb7afe2f7fb836e59fe605d30c88b3b5b13a6d8ab2ec0...
2,version_diagnostico,GIAD-M3-S4-IOT-diagnostico-v4
3,filas_bronze,28920
4,filas_diagnosticadas,28920
5,filas_en_cuarentena,554
6,filas_sin_cuarentena,28366
7,problemas_detectados,560
8,camion_id:FORMATO_INVALIDO,50
9,humedad_cabina_pct:FALTANTE,100


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales en `S4/andinalog_iot_telemetry/notebook1/salidas/` y se reemplazan con el mismo nombre al final. El CSV Bronze nunca se sobrescribe. Si se vuelve a ejecutar con la misma fuente y reglas, las salidas se actualizan en lugar de acumular versiones antiguas.


In [11]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_iot_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_iot_telemetry_diagnosticado.csv": df_diagnosticado,
    "andinalog_iot_telemetry_problemas.csv": df_problemas,
    "andinalog_iot_telemetry_cuarentena.csv": df_cuarentena,
    "andinalog_iot_telemetry_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_iot_telemetry\notebook1\salidas\andinalog_iot_telemetry_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_iot_telemetry\notebook1\salidas\andinalog_iot_telemetry_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_iot_telemetry\notebook1\salidas\andinalog_iot_telemetry_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_iot_telemetry\notebook1\salidas\andinalog_iot_telemetry_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. El informe de S4 justifica tratamientos posibles, pero ninguna regla de curación está aprobada automáticamente por este diagnóstico. Una fila saldrá de cuarentena solo cuando todos sus problemas hayan sido resueltos y validados.
